# UTS Machine Learning: Analisis & Klasifikasi Komentar YouTube

Notebook ini berisi pipeline end-to-end:
1. **Scraping Data Komentar**: Mengambil komentar video YouTube menggunakan YouTube Data API v3 (`comments_raw.csv`).
2. **Persiapan Data Anotasi**: Memilih sampel data untuk pelabelan manual (`comments_to_label.csv`).
3. **Data Cleaning & Text Preprocessing**: Case folding, regex cleansing, tokenization, stopword removal, dan stemming Bahasa Indonesia (Sastrawi).
4. **Exploratory Data Analysis (EDA)**: Analisis distribusi kelas, panjang teks, dan visualisasi kata kunci.
5. **Feature Extraction**: Pembobotan kata dengan TF-IDF (Term Frequency - Inverse Document Frequency).
6. **Model Training & Comparison**: Melatih algoritma klasifikasi (Naive Bayes, SVM, Logistic Regression).
7. **Evaluasi & Pengujian**: Confusion matrix, Classification report (Accuracy, Precision, Recall, F1), dan fungsi prediksi komentar baru.

## 1. Import Library & Setup Environment

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from googleapiclient.discovery import build

# NLP & Preprocessing
import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load konfigurasi dari file .env
load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY", "")
VIDEO_ID = os.getenv("VIDEO_ID", "")
MAX_RESULTS = int(os.getenv("MAX_RESULTS", 1000))

print("Environment loaded successfully.")
if not YOUTUBE_API_KEY:
    print("⚠️ PERINGATAN: YOUTUBE_API_KEY belum diisi di file .env")
if not VIDEO_ID:
    print("ℹ️ VIDEO_ID belum diisi di .env. Anda dapat mengisinya sebelum scraping.")

## 2. Scraping Komentar YouTube (YouTube Data API v3)

Fungsi ini akan mengambil komentar tingkat atas (top-level comments) dari video target dan menyimpannya ke `comments_raw.csv`.

In [ ]:
def scrape_youtube_comments(video_id: str, api_key: str, max_comments: int = 1000, output_file: str = "comments_raw.csv") -> pd.DataFrame:
    """
    Mengambil komentar dari YouTube Data API v3 dan menyimpannya ke file CSV.
    """
    if not api_key:
        raise ValueError("YOUTUBE_API_KEY wajib diisi untuk menjalankan scraping.")
    if not video_id:
        raise ValueError("VIDEO_ID wajib diisi untuk menentukan video target.")

    youtube = build("youtube", "v3", developerKey=api_key)
    comments_data = []
    next_page_token = None

    print(f"Memulai scraping komentar untuk Video ID: {video_id}...")

    while len(comments_data) < max_comments:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(100, max_comments - len(comments_data)),
            pageToken=next_page_token,
            textFormat="plainText",
            order="relevance"
        )
        response = request.execute()

        for item in response.get("items", []):
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            comments_data.append({
                "comment_id": item["id"],
                "author": snippet.get("authorDisplayName", ""),
                "comment_text": snippet.get("textDisplay", ""),
                "like_count": snippet.get("likeCount", 0),
                "published_at": snippet.get("publishedAt", ""),
                "updated_at": snippet.get("updatedAt", "")
            })

        next_page_token = response.get("nextPageToken")
        print(f"Berhasil mengumpulkan {len(comments_data)} komentar...")
        
        if not next_page_token:
            print("Tidak ada komentar lagi untuk diambil.")
            break

    df_raw = pd.DataFrame(comments_data)
    df_raw.to_csv(output_file, index=False, encoding="utf-8")
    print(f"✅ Scraping selesai! Total {len(df_raw)} komentar disimpan ke '{output_file}'.")
    return df_raw

# Uncomment baris di bawah ini ketika Anda siap memulai scraping:
# df_raw = scrape_youtube_comments(VIDEO_ID, YOUTUBE_API_KEY, max_comments=MAX_RESULTS)

## 3. Persiapan Data Anotasi / Labeling (`comments_to_label.csv`)

Setelah `comments_raw.csv` diperoleh, kita dapat membuat subset komentar yang siap dianotasi (misalnya: Positif, Negatif, atau Netral).

In [ ]:
def prepare_labeling_dataset(raw_csv_path: str = "comments_raw.csv", output_csv_path: str = "comments_to_label.csv", sample_size: int = 500) -> pd.DataFrame:
    """
    Mempersiapkan template file komentar untuk proses labeling manual.
    """
    if not os.path.exists(raw_csv_path):
        print(f"ℹ️ File '{raw_csv_path}' belum ditemukan. Jalankan scraping terlebih dahulu.")
        return None

    df = pd.read_csv(raw_csv_path)
    df_clean = df.drop_duplicates(subset=["comment_text"]).dropna(subset=["comment_text"])
    
    sample_n = min(sample_size, len(df_clean))
    df_sample = df_clean.sample(n=sample_n, random_state=42)[["comment_id", "author", "comment_text", "like_count"]].copy()
    df_sample["label"] = ""  # Kolom untuk diisi anotasi: Positif / Negatif / Netral
    
    df_sample.to_csv(output_csv_path, index=False, encoding="utf-8")
    print(f"✅ Berhasil menyiapkan {len(df_sample)} sampel komentar untuk anotasi di '{output_csv_path}'.")
    return df_sample

# Uncomment baris di bawah ini setelah memiliki comments_raw.csv:
# df_to_label = prepare_labeling_dataset(sample_size=500)

## 4. Text Preprocessing (Cleansing, Stopwords, Stemming)

Tahapan pembersihan teks komentar YouTube:
- **Case Folding**: Mengubah semua teks menjadi huruf kecil.
- **Regex Cleansing**: Menghapus URL, hashtag, mention, angka, dan karakter khusus.
- **Stopwords Removal**: Menghapus kata umum yang tidak memiliki bobot makna signifikan.
- **Stemming**: Mengubah kata berimbuhan menjadi kata dasar menggunakan `PySastrawi`.

In [ ]:
# Inisialisasi Stopword Remover dan Stemmer Sastrawi
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 1. Lowercase
    text = text.lower()
    # 2. Hapus URL
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    # 3. Hapus mention (@user) dan hashtag (#tag)
    text = re.sub(r"[@#]\S+", "", text)
    # 4. Hapus karakter non-alfabet (angka & tanda baca)
    text = re.sub(r"[^a-z\s]", " ", text)
    # 5. Hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_text(text: str) -> str:
    cleaned = clean_text(text)
    without_stopwords = stopword_remover.remove(cleaned)
    stemmed = stemmer.stem(without_stopwords)
    return stemmed

# Contoh demonstrasi preprocessing
sample_comment = "Wah keren banget videonya!! Sangat bermanfaat untuk belajar ML @admin https://youtube.com/"
print("Sebelum:", sample_comment)
print("Sesudah:", preprocess_text(sample_comment))

## 5. Memuat Data Berlabel & Exploratory Data Analysis (EDA)

Bagian ini membaca dataset yang sudah dilabeli (misalnya `comments_to_label.csv` setelah selesai dianotasi) untuk dianalisis dan dimodelkan.

In [ ]:
data_path = "comments_to_label.csv"

if os.path.exists(data_path):
    df_labeled = pd.read_csv(data_path)
    print(f"Data berlabel berhasil dimuat: {len(df_labeled)} baris.")
    display(df_labeled.head())
    
    # Visualisasi distribusi kelas label
    if "label" in df_labeled.columns and df_labeled["label"].notna().any():
        plt.figure(figsize=(7, 4))
        sns.countplot(data=df_labeled, x="label", palette="viridis")
        plt.title("Distribusi Label Sentimen Komentar")
        plt.xlabel("Label")
        plt.ylabel("Jumlah Komentar")
        plt.show()
else:
    print(f"ℹ️ '{data_path}' belum tersedia. Selesaikan scraping dan anotasi data terlebih dahulu.")

## 6. Feature Extraction (TF-IDF Vectorization)

In [ ]:
# Inisialisasi TfidfVectorizer dengan unigram & bigram
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=2500,
    min_df=2
)

## 7. Model Training & Comparison

Mengevaluasi dan membandingkan performa model Machine Learning:
1. **Multinomial Naive Bayes**
2. **Support Vector Machine (LinearSVC)**
3. **Logistic Regression**

In [ ]:
def train_and_evaluate_models(df: pd.DataFrame, text_col: str = "comment_text", label_col: str = "label"):
    """
    Melatih dan membandingkan beberapa model Machine Learning pada dataset berlabel.
    """
    df_valid = df.dropna(subset=[text_col, label_col])
    df_valid = df_valid[df_valid[label_col].astype(str).str.strip() != ""]
    
    if len(df_valid) == 0:
        print("Data berlabel kosong!")
        return

    print("Memulai preprocessing teks...")
    X_clean = df_valid[text_col].apply(preprocess_text)
    y = df_valid[label_col]

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_clean, y, test_size=0.2, random_state=42, stratify=y
    )

    X_train_vec = tfidf_vectorizer.fit_transform(X_train_raw)
    X_test_vec = tfidf_vectorizer.transform(X_test_raw)

    models = {
        "Multinomial Naive Bayes": MultinomialNB(),
        "Linear SVM": LinearSVC(random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_test_vec)
        acc = accuracy_score(y_test, y_pred)
        results[name] = {"model": model, "accuracy": acc, "pred": y_pred}
        print(f"\n{'='*50}\nModel: {name} (Akurasi: {acc:.4f})\n{'='*50}")
        print(classification_report(y_test, y_pred))

    return results, X_test_raw, y_test

# Jalankan fungsi ini saat dataset berlabel sudah siap:
# if os.path.exists(data_path) and 'df_labeled' in locals():
#     results, X_test, y_test = train_and_evaluate_models(df_labeled)

## 8. Pengujian Inferensi Komentar Baru

In [ ]:
def predict_comment_sentiment(text: str, trained_model, vectorizer) -> str:
    """
    Memprediksi sentimen untuk 1 teks komentar baru.
    """
    preprocessed = preprocess_text(text)
    vec = vectorizer.transform([preprocessed])
    pred = trained_model.predict(vec)
    return pred[0]

# Contoh pemanggilan inferensi:
# sample_new_comment = "Penjelasannya sangat mudah dipahami, terima kasih banyak!"
# print("Prediksi sentimen:", predict_comment_sentiment(sample_new_comment, best_model, tfidf_vectorizer))